In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 20:03:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/08 20:03:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/08 20:03:04 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


SparkSession siap. Versi Spark: 3.5.9


In [3]:
!hdfs dfs -mkdir -p /user/zeno/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/zeno/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/zeno/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/zeno/tugas7/raw/

In [4]:
df_cabang = spark.read.json("hdfs://localhost:9000/user/zeno/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/zeno/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/zeno/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)

print("Hasil Extract")
print(f"Cabang: {df_cabang.count()} | Menu: {df_menu.count()} | Transaksi: {df_transaksi.count()}")

Hasil Extract
Cabang: 3 | Menu: 15 | Transaksi: 4000


In [5]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                           .join(df_menu, on="menu_id", how="inner") \
                           .withColumn("total_penjualan", col("qty") * col("harga"))

print("\n--- Hasil Join & Kolom Turunan ---")
df_gabungan.show(5)


--- Hasil Join & Kolom Turunan ---
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Ma

In [6]:
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Window Function untuk mengambil top-2 menu per cabang
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                 .filter(col("peringkat") <= 2)

print("\n--- Top-2 Menu Terlaris per Cabang ---")
top2_menu.orderBy("nama_cabang", "peringkat").show()


--- Top-2 Menu Terlaris per Cabang ---
+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



In [7]:
df_gabungan.createOrReplaceTempView("tugas7_gabungan")

penjualan_barista = spark.sql("""
    SELECT 
        kepala_barista, 
        SUM(total_penjualan) AS total_penjualan
    FROM tugas7_gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

print("\n--- Total Penjualan per Kepala Barista (Spark SQL) ---")
penjualan_barista.show()


--- Total Penjualan per Kepala Barista (Spark SQL) ---
+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [8]:
path_output = "hdfs://localhost:9000/user/zeno/tugas7/processed/laporan"

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(path_output)

df_verifikasi = spark.read.parquet(path_output)
print("\n--- Verifikasi Load Parquet ---")
print(f"Data berhasil disimpan dan dibaca kembali. Total baris: {df_verifikasi.count()}")


--- Verifikasi Load Parquet ---
Data berhasil disimpan dan dibaca kembali. Total baris: 4000


In [9]:
ringkasan_eksekutif_tugas = df_gabungan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

print("\n--- Tabel Ringkasan Eksekutif ---")
ringkasan_eksekutif_tugas.show()


--- Tabel Ringkasan Eksekutif ---
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



Berdasarkan hasil pengolahan data end-to-end yang telah dilakukan, dapat disimpulkan bahwa Kopi Nusantara Semarang mencatatkan kinerja operasional terbaik. Cabang ini memimpin perolehan pendapatan dengan total omzet penjualan tertinggi, jumlah transaksi terbanyak, serta nilai average basket size yang sangat stabil. Pencapaian unggul ini didukung oleh tingginya volume transaksi harian serta popularitas produk di cabang tersebut.   Sebaliknya, cabang dengan performa terlemah berada di Kopi Nusantara Yogyakarta. Meskipun rata-rata nilai transaksi per perlanggan relatif bersaing, cabang ini memiliki total volume transaksi dan omzet keseluruhan yang paling rendah dibandingkan cabang lainnya.   Untuk meningkatkan performa cabang Yogyakarta, manajemen disarankan untuk menggalakkan promosi gencar pada Menu-13 dan Menu-2. Berdasarkan analisis window function, kedua menu ini terbukti menjadi menu terlaris (top-2) di cabang Yogyakarta yang menyumbang porsi pendapatan terbesar. Dengan menjadikan Menu-13 dan Menu-2 sebagai produk unggulan dalam penawaran paket bundling hemat atau diskon jam sibuk (happy hour), Kopi Nusantara Yogyakarta dapat menarik lebih banyak volume pengunjung harian dan mendorong daya beli pelanggan secara signifikan. 